<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/notebook_9_XGBoost_LightGBM_CatBoost_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install workalendar -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 49.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.0 MB/s eta 0:00:00


In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 8.7 MB/s eta 0:00:00


In [ ]:
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
from workalendar.europe import Russia
from tqdm import tqdm
import os

In [ ]:
def save_predictions(ts, house_ids, y_true, y_pred, horizon_name, model_name, filename):
    df_pred = pd.DataFrame({
        'timestamp': ts,
        'house_id': house_ids,
        'y_true': y_true,
        'y_pred': y_pred,
        'horizon': horizon_name,
        'model': model_name,
    })
    if os.path.exists(filename):
        df_existing = pd.read_csv(filename)
        df_pred = pd.concat([df_existing, df_pred], ignore_index=True)
    df_pred.to_csv(filename, index=False)

In [ ]:
df = pd.read_csv('df_final+whether.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

HOUSE_META = {
    'house_1': {'n_flats': 383, 'n_floors': 12},
    'house_2': {'n_flats': 191, 'n_floors': 12},
    'house_3': {'n_flats': 124, 'n_floors': 12},
    'house_4': {'n_flats': 263, 'n_floors': 12},
    'house_5': {'n_flats': 127, 'n_floors':  7},
    'house_6': {'n_flats': 497, 'n_floors': 25},
    'house_7': {'n_flats': 471, 'n_floors': 17},
    'house_8': {'n_flats': 171, 'n_floors': 23},
}
HOUSES = list(HOUSE_META.keys())

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df['is_holiday'] = df['timestamp'].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {'MAE': round(mae, 3), 'RMSE': round(rmse, 3), 'MAPE': round(mape, 3)}

horizons = {
    '4h': 8,
    '8h': 16,
    '24h': 48,
    '7d': 336,
    '14d': 672,
    '1m': 1488,
}

# split по времени
split_train = df['timestamp'].quantile(0.70)
split_val = df['timestamp'].quantile(0.85)

# фолды по уникальным timestamps
unique_ts = df['timestamp'].values
tscv = TimeSeriesSplit(n_splits=5)
folds = []
for train_idx, val_idx in tscv.split(unique_ts):
    folds.append((unique_ts[train_idx[-1]], unique_ts[val_idx[-1]]))

for i, (t_tr, t_vl) in enumerate(folds):
    print(f'  Фолд {i+1}: train до {pd.Timestamp(t_tr).date()} | val до {pd.Timestamp(t_vl).date()}')

  Фолд 1: train до 2017-12-14 | val до 2018-01-17
  Фолд 2: train до 2018-01-17 | val до 2018-02-20
  Фолд 3: train до 2018-02-20 | val до 2018-03-25
  Фолд 4: train до 2018-03-25 | val до 2018-04-28
  Фолд 5: train до 2018-04-28 | val до 2018-06-01


In [ ]:
def make_features_all(df, lag_features=[1, 2, 48, 96, 336]):
    frames = []
    for house, meta in HOUSE_META.items():
        data = df[['timestamp', house]].copy()
        data = data.rename(columns={house: 'power'})
        data['hour'] = data['timestamp'].dt.hour
        data['minute'] = data['timestamp'].dt.minute
        data['weekday'] = data['timestamp'].dt.weekday
        data['month'] = data['timestamp'].dt.month
        data['day_of_year'] = data['timestamp'].dt.dayofyear
        data['is_weekend'] = (data['weekday'] >= 5).astype(int)
        data['is_holiday'] = df['is_holiday'].values
        for lag in lag_features:
            data[f'lag_{lag}'] = data['power'].shift(lag)
        data['rolling_mean_48'] = data['power'].shift(1).rolling(48).mean()
        data['rolling_mean_336'] = data['power'].shift(1).rolling(336).mean()
        data['temp_c'] = df['temp_c'].values
        data['humidity'] = df['humidity'].values
        data['cloudiness'] = df['cloudiness'].values
        data['n_flats'] = meta['n_flats']
        data['n_floors'] = meta['n_floors']
        data['house_id'] = house
        frames.append(data)

    df_all = pd.concat(frames, ignore_index=True)
    df_all = df_all.sort_values(['timestamp', 'house_id']).reset_index(drop=True)
    df_all = df_all.dropna().reset_index(drop=True)
    return df_all

df_long = make_features_all(df)

feature_cols = [c for c in df_long.columns
                if c not in ['timestamp', 'power', 'house_id']]

print(f'Строк: {len(df_long)}')
print(f'Признаки: {feature_cols}')

Строк: 75008
Признаки: ['hour', 'minute', 'weekday', 'month', 'day_of_year', 'is_weekend', 'is_holiday', 'lag_1', 'lag_2', 'lag_48', 'lag_96', 'lag_336', 'rolling_mean_48', 'rolling_mean_336', 'temp_c', 'humidity', 'cloudiness', 'n_flats', 'n_floors']


In [ ]:
import joblib, os
os.makedirs('models', exist_ok=True)

xgb_rows = []

for horizon_name, horizon_points in tqdm(horizons.items(), desc='XGBoost'):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=['power_target']).reset_index(drop=True)
    ts_shifted = df_shifted['timestamp']

    fold_metrics_all = []

    for t_train_end, t_val_end in folds:
        mask_tr = ts_shifted <= t_train_end
        X_tr = df_shifted.loc[mask_tr, feature_cols]
        y_tr = df_shifted.loc[mask_tr, 'power_target']

        if X_tr.empty:
            continue

        model = XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbosity=0
        )
        model.fit(X_tr, y_tr)

        for house in HOUSES:
            mask_h = (ts_shifted > t_train_end) & \
                     (ts_shifted <= t_val_end) & \
                     (df_shifted['house_id'] == house)
            X_vl_h = df_shifted.loc[mask_h, feature_cols].head(horizon_points)
            y_vl_h = df_shifted.loc[mask_h, 'power_target'].head(horizon_points)

            if len(y_vl_h) < horizon_points:
                continue

            y_pred_h = model.predict(X_vl_h)
            fold_metrics_all.append({
                'house': house,
                **evaluate(y_vl_h.values, y_pred_h)
            })

            if t_train_end == folds[-1][0]:
              save_predictions(
                ts=df_shifted.loc[mask_h, 'timestamp'].head(horizon_points).values,
                house_ids=np.full(len(y_vl_h), house),
                y_true=y_vl_h.values,
                y_pred=y_pred_h,
                horizon_name=horizon_name,
                model_name='XGBoost',
                filename='predictions_xgb.csv'
              )

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold['house'] == house]
        if len(m) == 0:
            continue
        xgb_rows.append({
            'модель': 'XGBoost', 'дом': house, 'горизонт': horizon_name,
            'MAE':  round(m['MAE'].mean(), 3),
            'RMSE': round(m['RMSE'].mean(), 3),
            'MAPE': round(m['MAPE'].mean(), 3),
        })

df_xgb = pd.DataFrame(xgb_rows)
df_xgb.to_csv('results_xgboost.csv', index=False)

print('XGBoost: MAPE:')
pivot = df_xgb.pivot(index='горизонт', columns='дом', values='MAPE')
pivot = pivot.reindex(list(horizons.keys()))
pivot['Среднее'] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

XGBoost: 100%|██████████| 6/6 [02:09<00:00, 21.65s/it]

XGBoost: MAPE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h           5.49     5.62     5.23     4.09     5.45     3.23     4.39     5.42     4.87
8h           5.18     7.57     8.04     5.31     4.87     6.21     5.42     7.78     6.30
24h          7.10     8.53     9.15     7.45     6.19     5.40     5.64     6.71     7.02
7d           5.63     7.13     8.60     6.56     7.91     5.89     6.86     6.32     6.86
14d          7.79     8.65     9.59     9.67     9.14     9.61     7.57     7.68     8.71
1m           9.12     9.80    11.79    10.48    10.38    11.73     8.55     8.51    10.04


In [ ]:
lgbm_rows = []
lgbm_final_models = {}

for horizon_name, horizon_points in tqdm(horizons.items(), desc='LightGBM'):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=['power_target']).reset_index(drop=True)
    ts_shifted = df_shifted['timestamp']

    fold_metrics_all = []

    for t_train_end, t_val_end in folds:
        mask_tr = ts_shifted <= t_train_end
        X_tr = df_shifted.loc[mask_tr, feature_cols]
        y_tr = df_shifted.loc[mask_tr, 'power_target']

        if X_tr.empty:
            continue

        model = LGBMRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbose=-1
        )
        model.fit(X_tr, y_tr)

        for house in HOUSES:
            mask_h = (ts_shifted > t_train_end) & \
                     (ts_shifted <= t_val_end) & \
                     (df_shifted['house_id'] == house)
            X_vl_h = df_shifted.loc[mask_h, feature_cols].head(horizon_points)
            y_vl_h = df_shifted.loc[mask_h, 'power_target'].head(horizon_points)

            if len(y_vl_h) < horizon_points:
                continue

            y_pred_h = model.predict(X_vl_h)
            fold_metrics_all.append({
                'house': house,
                **evaluate(y_vl_h.values, y_pred_h)
            })

            if t_train_end == folds[-1][0]:
              save_predictions(
                ts=df_shifted.loc[mask_h, 'timestamp'].head(horizon_points).values,
                house_ids=np.full(len(y_vl_h), house),
                y_true=y_vl_h.values,
                y_pred=y_pred_h,
                horizon_name=horizon_name,
                model_name='LightGBM',
                filename='predictions_lgbm.csv'
              )

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold['house'] == house]
        if len(m) == 0:
            continue
        lgbm_rows.append({
            'модель': 'LightGBM', 'дом': house, 'горизонт': horizon_name,
            'MAE':  round(m['MAE'].mean(), 3),
            'RMSE': round(m['RMSE'].mean(), 3),
            'MAPE': round(m['MAPE'].mean(), 3),
        })

    # финальная модель на полном train
    mask_full_tr = ts_shifted <= split_train
    X_full_tr = df_shifted.loc[mask_full_tr, feature_cols]
    y_full_tr = df_shifted.loc[mask_full_tr, 'power_target']

    final_model = LGBMRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )
    final_model.fit(X_full_tr, y_full_tr)
    lgbm_final_models[horizon_name] = final_model
    joblib.dump(final_model, f'models/lgbm_{horizon_name}.pkl')
    print(f'  Сохранено: models/lgbm_{horizon_name}.pkl')

df_lgbm = pd.DataFrame(lgbm_rows)
df_lgbm.to_csv('results_lgbm.csv', index=False)

print('\nLightGBM: MAPE:')
pivot = df_lgbm.pivot(index='горизонт', columns='дом', values='MAPE')
pivot = pivot.reindex(list(horizons.keys()))
pivot['Среднее'] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

LightGBM:  17%|█▋        | 1/6 [00:13<01:09, 13.90s/it]

  Сохранено: models/lgbm_4h.pkl


LightGBM:  33%|███▎      | 2/6 [00:27<00:54, 13.58s/it]

  Сохранено: models/lgbm_8h.pkl


LightGBM:  50%|█████     | 3/6 [00:40<00:40, 13.64s/it]

...

  Сохранено: models/lgbm_7d.pkl


LightGBM:  83%|████████▎ | 5/6 [01:10<00:14, 14.25s/it]

  Сохранено: models/lgbm_14d.pkl


LightGBM: 100%|██████████| 6/6 [01:25<00:00, 14.19s/it]

  Сохранено: models/lgbm_1m.pkl

LightGBM: MAPE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h           5.42     5.67     5.85     3.40     5.26     3.11     4.33     6.02     4.88
8h           5.80     7.94     8.48     5.11     5.34     7.35     5.37     7.59     6.62
24h          7.06     8.48     9.48     7.36     6.19     5.49     5.67     6.82     7.07
7d           5.83     6.95     8.52     6.79     7.78     5.73     7.44     6.40     6.93
14d          7.97     8.80     9.80     9.99     9.16     9.86     7.47     7.94     8.87
1m           9.30    10.05    12.19    10.50    10.32    11.69     8.56     8.56    10.15


In [ ]:
meta = {
    'houses':  HOUSES,
    'horizons':  horizons,
    'house_meta': HOUSE_META,
    'feature_cols': feature_cols,
    'split_train': str(split_train),
    'split_val': str(split_val),
}
joblib.dump(meta, 'models/model_meta.pkl')

['models/model_meta.pkl']

In [ ]:
t_train_end, t_val_end = folds[-1]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Горизонт {h}' for h in horizons.keys()],
    vertical_spacing=0.1, horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    frames_h = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_h.append(df_h)
    df_h_all = pd.concat(frames_h, ignore_index=True)
    df_h_all = df_h_all.dropna(subset=['power_target']).reset_index(drop=True)
    ts_h = df_h_all['timestamp']

    mask_tr = ts_h <= t_train_end
    X_tr = df_h_all.loc[mask_tr, feature_cols]
    y_tr = df_h_all.loc[mask_tr, 'power_target']

    model_xgb_viz = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    )
    model_xgb_viz.fit(X_tr, y_tr)

    mask_vl = (ts_h > t_train_end) & \
              (ts_h <= t_val_end) & \
              (df_h_all['house_id'] == 'house_1')
    X_vl = df_h_all.loc[mask_vl, feature_cols].head(horizon_points)
    y_vl = df_h_all.loc[mask_vl, 'power_target'].head(horizon_points)
    ts_vl = df_h_all.loc[mask_vl, 'timestamp'].head(horizon_points)

    if len(y_vl) < horizon_points:
        continue

    y_pred = model_xgb_viz.predict(X_vl)

    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_vl.values, mode='lines', name='факт',
        line=dict(color='#1f77b4', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_pred, mode='lines', name='прогноз XGBoost',
        line=dict(color='#d62728', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title='XGBoost: прогнозные и фактические значения по всем горизонтам (дом 1, fold 5)',
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40), legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image('30_xgb_forecast_all_horizons.png', scale=2)

In [ ]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Горизонт {h}' for h in horizons.keys()],
    vertical_spacing=0.1, horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    frames_h = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_h.append(df_h)
    df_h_all = pd.concat(frames_h, ignore_index=True)
    df_h_all = df_h_all.dropna(subset=['power_target']).reset_index(drop=True)
    ts_h = df_h_all['timestamp']

    mask_tr = ts_h <= t_train_end
    X_tr = df_h_all.loc[mask_tr, feature_cols]
    y_tr = df_h_all.loc[mask_tr, 'power_target']

    model_lgbm_viz = LGBMRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )
    model_lgbm_viz.fit(X_tr, y_tr)

    mask_vl = (ts_h > t_train_end) & \
              (ts_h <= t_val_end) & \
              (df_h_all['house_id'] == 'house_1')
    X_vl = df_h_all.loc[mask_vl, feature_cols].head(horizon_points)
    y_vl = df_h_all.loc[mask_vl, 'power_target'].head(horizon_points)
    ts_vl = df_h_all.loc[mask_vl, 'timestamp'].head(horizon_points)

    if len(y_vl) < horizon_points:
        continue

    y_pred = model_lgbm_viz.predict(X_vl)

    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_vl.values, mode='lines', name='факт',
        line=dict(color='#1f77b4', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_pred, mode='lines', name='прогноз LightGBM',
        line=dict(color='#d62728', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title='LightGBM: прогнозные и фактические значения по всем горизонтам (дом 1, fold 5)',
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40), legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image('31_lgbm_forecast_all_horizons.png', scale=2)

In [ ]:
# важность признаков, горизонт 24h
model_xgb_fi  = model_xgb_viz   # последняя обученная XGBoost модель
model_lgbm_fi = lgbm_final_models['24h']

importances_xgb  = model_xgb_fi.feature_importances_
importances_lgbm = model_lgbm_fi.feature_importances_

idx_xgb  = np.argsort(importances_xgb)[::-1]
idx_lgbm = np.argsort(importances_lgbm)[::-1]

fig = make_subplots(rows=1, cols=2, subplot_titles=('XGBoost', 'LightGBM'))

fig.add_trace(go.Bar(
    x=[feature_cols[i] for i in idx_xgb],
    y=importances_xgb[idx_xgb],
    marker_color='#d62728', showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=[feature_cols[i] for i in idx_lgbm],
    y=importances_lgbm[idx_lgbm],
    marker_color='#1f77b4', showlegend=False
), row=1, col=2)

fig.update_layout(
    title='Важность признаков: горизонт 24h (общая модель)',
    width=900, height=400, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=80)
)
fig.update_xaxes(tickangle=45)
fig.show()
fig.write_image('32_feature_importance.png', scale=2)

In [ ]:
#CatBoost
from catboost import CatBoostRegressor

catboost_rows = []

for horizon_name, horizon_points in tqdm(horizons.items(), desc='CatBoost'):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=['power_target']).reset_index(drop=True)
    ts_shifted = df_shifted['timestamp']

    fold_metrics_all = []

    for t_train_end, t_val_end in folds:
        mask_tr = ts_shifted <= t_train_end
        X_tr = df_shifted.loc[mask_tr, feature_cols]
        y_tr = df_shifted.loc[mask_tr, 'power_target']

        if X_tr.empty:
            continue

        model = CatBoostRegressor(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            subsample=0.8,
            random_seed=42,
            verbose=0
        )
        model.fit(X_tr, y_tr)

        for house in HOUSES:
            mask_h = (ts_shifted > t_train_end) & \
                     (ts_shifted <= t_val_end) & \
                     (df_shifted['house_id'] == house)
            X_vl_h = df_shifted.loc[mask_h, feature_cols].head(horizon_points)
            y_vl_h = df_shifted.loc[mask_h, 'power_target'].head(horizon_points)
            ts_vl_h = df_shifted.loc[mask_h, 'timestamp'].head(horizon_points)

            if len(y_vl_h) < horizon_points:
                continue

            y_pred_h = model.predict(X_vl_h)
            fold_metrics_all.append({
                'house': house,
                **evaluate(y_vl_h.values, y_pred_h)
            })

            if t_train_end == folds[-1][0]:
                save_predictions(
                    ts=ts_vl_h.values,
                    house_ids=np.full(len(y_vl_h), house),
                    y_true=y_vl_h.values,
                    y_pred=y_pred_h,
                    horizon_name=horizon_name,
                    model_name='CatBoost',
                    filename='predictions_catboost.csv'
                )

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold['house'] == house]
        if len(m) == 0:
            continue
        catboost_rows.append({
            'модель': 'CatBoost', 'дом': house, 'горизонт': horizon_name,
            'MAE':  round(m['MAE'].mean(), 3),
            'RMSE': round(m['RMSE'].mean(), 3),
            'MAPE': round(m['MAPE'].mean(), 3),
        })

df_catboost = pd.DataFrame(catboost_rows)
df_catboost.to_csv('results_catboost.csv', index=False)

print('CatBoost: MAPE:')
pivot = df_catboost.pivot(index='горизонт', columns='дом', values='MAPE')
pivot = pivot.reindex(list(horizons.keys()))
pivot['Среднее'] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

CatBoost: 100%|██████████| 6/6 [02:52<00:00, 28.81s/it]

CatBoost: MAPE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h           4.92     6.22     6.68     3.97     6.39     4.68     4.23     6.16     5.41
8h           5.33     7.84     7.57     4.76     5.57     5.73     5.46     4.42     5.83
24h          7.08     8.07     9.38     7.47     6.07     5.51     5.38     6.62     6.95
7d           5.35     7.20     8.48     6.54     7.53     5.67     5.99     6.30     6.64
14d          7.71     8.98     9.80     9.86     9.45     9.92     7.57     7.86     8.90
1m           8.80     9.36    11.15    10.08     9.94    11.98     9.08     8.76     9.89


In [ ]:
t_train_end, t_val_end = folds[-1]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Горизонт {h}' for h in horizons.keys()],
    vertical_spacing=0.1, horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    frames_h = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_h.append(df_h)
    df_h_all = pd.concat(frames_h, ignore_index=True)
    df_h_all = df_h_all.dropna(subset=['power_target']).reset_index(drop=True)
    ts_h = df_h_all['timestamp']

    mask_tr = ts_h <= t_train_end
    X_tr = df_h_all.loc[mask_tr, feature_cols]
    y_tr = df_h_all.loc[mask_tr, 'power_target']

    model_viz = CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=6,
        subsample=0.8, random_seed=42, verbose=0
    )
    model_viz.fit(X_tr, y_tr)

    mask_vl = (ts_h > t_train_end) & \
              (ts_h <= t_val_end) & \
              (df_h_all['house_id'] == 'house_1')
    X_vl  = df_h_all.loc[mask_vl, feature_cols].head(horizon_points)
    y_vl  = df_h_all.loc[mask_vl, 'power_target'].head(horizon_points)
    ts_vl = df_h_all.loc[mask_vl, 'timestamp'].head(horizon_points)

    if len(y_vl) < horizon_points:
        continue

    y_pred = model_viz.predict(X_vl)

    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_vl.values, mode='lines', name='факт',
        line=dict(color='#1f77b4', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=ts_vl, y=y_pred, mode='lines', name='прогноз CatBoost',
        line=dict(color='#d62728', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

fig.update_layout(
    title='CatBoost: прогнозные и фактические значения по всем горизонтам (дом 1, fold 5)',
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40), legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image('33_catboost_forecast_all_horizons.png', scale=2)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_rows = []

for horizon_name, horizon_points in tqdm(horizons.items(), desc='RandomForest'):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=['power_target']).reset_index(drop=True)
    ts_shifted = df_shifted['timestamp']

    fold_metrics_all = []

    # только последний фолд
    for t_train_end, t_val_end in [folds[-1]]:
        mask_tr = ts_shifted <= t_train_end
        X_tr = df_shifted.loc[mask_tr, feature_cols]
        y_tr = df_shifted.loc[mask_tr, 'power_target']

        if X_tr.empty:
            continue

        # сэмплируем 30% train для ускорения счета
        sample_idx = X_tr.sample(frac=0.3, random_state=42).index
        X_tr_sample = X_tr.loc[sample_idx]
        y_tr_sample = y_tr.loc[sample_idx]

        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_tr_sample, y_tr_sample)

        for house in HOUSES:
            mask_h = (ts_shifted > t_train_end) & \
                     (ts_shifted <= t_val_end) & \
                     (df_shifted['house_id'] == house)
            X_vl_h  = df_shifted.loc[mask_h, feature_cols].head(horizon_points)
            y_vl_h  = df_shifted.loc[mask_h, 'power_target'].head(horizon_points)
            ts_vl_h = df_shifted.loc[mask_h, 'timestamp'].head(horizon_points)

            if len(y_vl_h) < horizon_points:
                continue

            y_pred_h = model.predict(X_vl_h)
            fold_metrics_all.append({
                'house': house,
                **evaluate(y_vl_h.values, y_pred_h)
            })

            # сохраняем предсказания
            save_predictions(
                ts=ts_vl_h.values,
                house_ids=np.full(len(y_vl_h), house),
                y_true=y_vl_h.values,
                y_pred=y_pred_h,
                horizon_name=horizon_name,
                model_name='RandomForest',
                filename='predictions_rf.csv'
            )

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold['house'] == house]
        if len(m) == 0:
            continue
        rf_rows.append({
            'модель': 'RandomForest', 'дом': house, 'горизонт': horizon_name,
            'MAE': round(m['MAE'].mean(), 3),
            'RMSE': round(m['RMSE'].mean(), 3),
            'MAPE': round(m['MAPE'].mean(), 3),
        })

df_rf = pd.DataFrame(rf_rows)
df_rf.to_csv('results_rf.csv', index=False)

print('RandomForest: MAPE:')
pivot = df_rf.pivot(index='горизонт', columns='дом', values='MAPE')
pivot = pivot.reindex(list(horizons.keys()))
pivot['Среднее'] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

RandomForest:  83%|████████▎ | 5/6 [01:20<00:16, 16.11s/it]


KeyError: 'house'

In [ ]:
df_pred_rf = pd.read_csv('predictions_rf.csv', parse_dates=['timestamp'])

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'Горизонт {h}' for h in horizons.keys()],
    vertical_spacing=0.1, horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    subset = df_pred_rf[
        (df_pred_rf['horizon'] == horizon_name) &
        (df_pred_rf['house_id'] == 'house_1')
    ].sort_values('timestamp').head(horizon_points)

    if len(subset) == 0:
        continue

    fig.add_trace(go.Scatter(
        x=subset['timestamp'], y=subset['y_true'],
        mode='lines', name='факт',
        line=dict(color='#1f77b4', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=subset['timestamp'], y=subset['y_pred'],
        mode='lines', name='прогноз RandomForest',
        line=dict(color='#d62728', width=1.2, shape='hv'),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text='Дата', row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text='кВт',  row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title='RandomForest: прогнозные и фактические значения по всем горизонтам (дом 1, fold 5)',
    width=700, height=900, font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40), legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image('34_rf_forecast_all_horizons.png', scale=2)

In [ ]:
lgbm_rows = []
lgbm_final_models = {}
lgbm_residuals = {}

for horizon_name, horizon_points in tqdm(horizons.items(), desc='LightGBM'):

    frames_shifted = []
    for house in HOUSES:
        df_h = df_long[df_long['house_id'] == house].copy()
        df_h['power_target'] = df_h['power'].shift(-horizon_points)
        frames_shifted.append(df_h)

    df_shifted = pd.concat(frames_shifted, ignore_index=True)
    df_shifted = df_shifted.dropna(subset=['power_target']).reset_index(drop=True)
    ts_shifted = df_shifted['timestamp']

    # пересчитываем split для текущего горизонта
    split_val_h   = pd.Timestamp(
        np.quantile(ts_shifted.astype(np.int64), 0.85)
    )
    split_train_h = pd.Timestamp(
        np.quantile(ts_shifted.astype(np.int64), 0.70)
    )

    fold_metrics_all = []

    for t_train_end, t_val_end in folds:
        mask_tr = ts_shifted <= t_train_end
        X_tr    = df_shifted.loc[mask_tr, feature_cols]
        y_tr    = df_shifted.loc[mask_tr, 'power_target']

        if X_tr.empty:
            continue

        model = LGBMRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbose=-1
        )
        model.fit(X_tr, y_tr)

        for house in HOUSES:
            mask_h  = (ts_shifted > t_train_end) & \
                      (ts_shifted <= t_val_end) & \
                      (df_shifted['house_id'] == house)
            X_vl_h  = df_shifted.loc[mask_h, feature_cols].head(horizon_points)
            y_vl_h  = df_shifted.loc[mask_h, 'power_target'].head(horizon_points)
            ts_vl_h = df_shifted.loc[mask_h, 'timestamp'].head(horizon_points)

            if len(y_vl_h) < horizon_points:
                continue

            y_pred_h = model.predict(X_vl_h)
            fold_metrics_all.append({
                'house': house,
                **evaluate(y_vl_h.values, y_pred_h)
            })

            if t_train_end == folds[-1][0]:
                save_predictions(
                    ts=ts_vl_h.values,
                    house_ids=np.full(len(y_vl_h), house),
                    y_true=y_vl_h.values,
                    y_pred=y_pred_h,
                    horizon_name=horizon_name,
                    model_name='LightGBM',
                    filename='predictions_lgbm.csv'
                )

    df_fold = pd.DataFrame(fold_metrics_all)
    for house in HOUSES:
        m = df_fold[df_fold['house'] == house]
        if len(m) == 0:
            continue
        lgbm_rows.append({
            'модель':   'LightGBM',
            'дом':      house,
            'горизонт': horizon_name,
            'MAE':      round(m['MAE'].mean(), 3),
            'RMSE':     round(m['RMSE'].mean(), 3),
            'MAPE':     round(m['MAPE'].mean(), 3),
        })

    # финальная модель на train+val (85% данных)
    mask_final = ts_shifted <= split_val_h
    X_final    = df_shifted.loc[mask_final, feature_cols]
    y_final    = df_shifted.loc[mask_final, 'power_target']

    if X_final.empty:
        continue

    final_model = LGBMRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )
    final_model.fit(X_final, y_final)
    lgbm_final_models[horizon_name] = final_model
    joblib.dump(final_model, f'models/lgbm_{horizon_name}.pkl')

    # остатки на тестовом периоде
    mask_test  = ts_shifted > split_val_h
    X_test     = df_shifted.loc[mask_test, feature_cols]
    y_test     = df_shifted.loc[mask_test, 'power_target']
    ts_test    = df_shifted.loc[mask_test, 'timestamp']
    house_test = df_shifted.loc[mask_test, 'house_id']

    if X_test.empty:
        continue

    y_pred_test = final_model.predict(X_test)
    residuals   = y_test.values - y_pred_test

    lgbm_residuals[horizon_name] = pd.DataFrame({
        'timestamp': ts_test.values,
        'house_id':  house_test.values,
        'y_true':    y_test.values,
        'y_pred':    y_pred_test,
        'residual':  residuals,
        'horizon':   horizon_name,
    })

# сохраняем остатки
df_residuals_all = pd.concat(lgbm_residuals.values(), ignore_index=True)
df_residuals_all.to_csv('models/lgbm_residuals.csv', index=False)

df_lgbm = pd.DataFrame(lgbm_rows)
df_lgbm.to_csv('results_lgbm.csv', index=False)

print('\nLightGBM: MAPE:')
pivot = df_lgbm.pivot(index='горизонт', columns='дом', values='MAPE')
pivot = pivot.reindex(list(horizons.keys()))
pivot['Среднее'] = pivot.mean(axis=1).round(2)
print(pivot.round(2).to_string())

# метаданные
meta = {
    'houses':       HOUSES,
    'horizons':     horizons,
    'house_meta':   HOUSE_META,
    'feature_cols': feature_cols,
    'split_train':  str(split_train),
    'split_val':    str(split_val),
}
joblib.dump(meta, 'models/model_meta.pkl')

LightGBM: 100%|██████████| 6/6 [04:38<00:00, 46.41s/it]



LightGBM: MAPE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8  Среднее
горизонт                                                                                 
4h           5.42     5.67     5.85     3.40     5.26     3.11     4.33     6.02     4.88
8h           5.80     7.94     8.48     5.11     5.34     7.35     5.37     7.59     6.62
24h          7.06     8.48     9.48     7.36     6.19     5.49     5.67     6.82     7.07
7d           5.83     6.95     8.52     6.79     7.78     5.73     7.44     6.40     6.93
14d          7.97     8.80     9.80     9.99     9.16     9.86     7.47     7.94     8.87
1m           9.30    10.05    12.19    10.50    10.32    11.69     8.56     8.56    10.15


['models/model_meta.pkl']